[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/practice/intermediate/01_from_scratch_algorithms.ipynb)

# Luyện tập trung cấp 1 — Tự cài thuật toán

Sáu thuật toán cài bằng NumPy, mỗi bài so khớp với scikit-learn. Viết lại từ trí nhớ (không mở chương) là cách kiểm tra tốt nhất bạn đã thực sự hiểu.

Chương liên quan: 06–11, 20–21.

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/practice/intermediate

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsRegressor

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_classification_2d

rng = np.random.default_rng(0)

**Bài 1 — Hồi quy logistic bằng gradient descent.** `fit_logistic(X, y, lr=0.5, n_iter=5000, l2=0.0)`: thêm cột 1, khởi tạo 0, gradient của log-loss trung bình (+ `l2 · w` cho mọi trọng số **trừ** hệ số chặn). Trả về `w` (phần tử đầu là hệ số chặn). Với `l2 = 1/(C·n)`, kết quả khớp `LogisticRegression(C=C)` của scikit-learn.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def fit_logistic(X, y, lr=0.5, n_iter=5000, l2=0.0):
    ### START CODE HERE ### (≈ 7 dòng)
    # STARTER: return np.zeros(X.shape[1] + 1)
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_iter):
        grad = Xb.T @ (sigmoid(Xb @ w) - y) / len(y)
        grad[1:] += l2 * w[1:]
        w -= lr * grad
    return w
    ### END CODE HERE ###


X1, y1 = make_classification_2d("blobs", n=300, noise=1.5, seed=1)
w = fit_logistic(X1, y1, l2=1 / (1.0 * len(y1)))
ref = LogisticRegression(C=1.0).fit(X1, y1)
print(w, ref.intercept_, ref.coef_)

In [ ]:
check_close(w, np.r_[ref.intercept_, ref.coef_[0]], tol=1e-3, message="khớp scikit-learn (cùng điều chuẩn L2)")

**Bài 2 — Decision stump.** `fit_stump(X, y)` → `(feature, threshold, left_value, right_value)` cho cây hồi quy độ sâu 1: thử mọi đặc trưng và mọi ngưỡng (trung điểm các giá trị khác nhau liên tiếp), chọn cái **tổng bình phương sai số** nhỏ nhất; giá trị lá là trung bình.

In [ ]:
def fit_stump(X, y):
    ### START CODE HERE ### (≈ 11 dòng)
    # STARTER: return 0, 0.0, 0.0, 0.0
    best = (None, None, None, None, np.inf)
    for j in range(X.shape[1]):
        values = np.unique(X[:, j])
        for t in (values[:-1] + values[1:]) / 2:
            left, right = y[X[:, j] <= t], y[X[:, j] > t]
            sse = ((left - left.mean()) ** 2).sum() + ((right - right.mean()) ** 2).sum()
            if sse < best[4]:
                best = (j, t, left.mean(), right.mean(), sse)
    return best[:4]
    ### END CODE HERE ###


X2 = rng.uniform(0, 10, (200, 3))
y2 = np.where(X2[:, 1] > 6, 5.0, 1.0) + rng.normal(0, 0.1, 200)
print(fit_stump(X2, y2))

In [ ]:
feature, threshold, lv, rv = fit_stump(X2, y2)
check_equal(feature, 1, "chọn đúng đặc trưng")
check_close(threshold, 6, tol=0.15, message="ngưỡng gần 6")
check_close([lv, rv], [1, 5], tol=0.05, message="giá trị lá")

**Bài 3 — kNN hồi quy.** `knn_regress(X_train, y_train, X_new, k)` = trung bình y của k láng giềng gần nhất (Euclid).

In [ ]:
def knn_regress(X_train, y_train, X_new, k):
    ### START CODE HERE ### (≈ 3 dòng)
    # STARTER: return np.zeros(len(X_new))
    d = ((X_new[:, None, :] - X_train[None, :, :]) ** 2).sum(-1)
    nearest = np.argsort(d, axis=1, kind="stable")[:, :k]
    return y_train[nearest].mean(axis=1)
    ### END CODE HERE ###


X3, X3_new = rng.normal(size=(150, 2)), rng.normal(size=(40, 2))
y3 = X3[:, 0] ** 2 + rng.normal(0, 0.1, 150)

In [ ]:
check_close(knn_regress(X3, y3, X3_new, 5), KNeighborsRegressor(5).fit(X3, y3).predict(X3_new), message="khớp scikit-learn")

**Bài 4 — K-Means đầy đủ.** `kmeans(X, k, n_iter=100, seed=0)`: khởi tạo bằng k điểm ngẫu nhiên (không lặp) của X, lặp gán–cập nhật đến khi tâm không đổi; trả về `(centers, labels, inertia)`. So inertia với scikit-learn (`n_init=10`): nghiệm của bạn (một lần khởi tạo) có thể **kém hơn** — lặp với nhiều seed và lấy inertia nhỏ nhất.

In [ ]:
def kmeans(X, k, n_iter=100, seed=0):
    ### START CODE HERE ### (≈ 11 dòng)
    # STARTER: return np.zeros((k, X.shape[1])), np.zeros(len(X), int), 0.0
    r = np.random.default_rng(seed)
    centers = X[r.choice(len(X), k, replace=False)]
    for _ in range(n_iter):
        labels = ((X[:, None, :] - centers[None]) ** 2).sum(-1).argmin(1)
        new = np.array([X[labels == j].mean(0) if np.any(labels == j) else centers[j] for j in range(k)])
        if np.allclose(new, centers):
            break
        centers = new
    labels = ((X[:, None, :] - centers[None]) ** 2).sum(-1).argmin(1)
    inertia = float(((X - centers[labels]) ** 2).sum())
    return centers, labels, inertia
    ### END CODE HERE ###


X4 = np.vstack([rng.normal(c, 0.5, (100, 2)) for c in [(0, 0), (4, 0), (2, 4), (6, 5)]])
best_inertia = min(kmeans(X4, 4, seed=s)[2] for s in range(10))
print(f"tốt nhất trong 10 seed: {best_inertia:.1f} | scikit-learn: {KMeans(4, n_init=10, random_state=0).fit(X4).inertia_:.1f}")

In [ ]:
check_close(best_inertia, KMeans(4, n_init=10, random_state=0).fit(X4).inertia_, tol=1e-6, message="nhiều khởi tạo → khớp scikit-learn")
check_equal(len(np.unique(kmeans(X4, 4)[1])), 4, "4 cụm")

**Bài 5 — PCA bằng ma trận hiệp phương sai.** `pca_eig(X, k)`: trừ trung bình, ma trận hiệp phương sai (chia n − 1), `np.linalg.eigh`, lấy k vector riêng có trị riêng lớn nhất → trả về `(components (k × d), explained_variance)` (trị riêng).

In [ ]:
def pca_eig(X, k):
    ### START CODE HERE ### (≈ 5 dòng)
    # STARTER: return np.zeros((k, X.shape[1])), np.zeros(k)
    Xc = X - X.mean(axis=0)
    cov = Xc.T @ Xc / (len(X) - 1)
    values, vectors = np.linalg.eigh(cov)
    order = np.argsort(values)[::-1][:k]
    return vectors[:, order].T, values[order]
    ### END CODE HERE ###


X5 = rng.normal(size=(300, 5)) @ rng.normal(size=(5, 5))
components, variances = pca_eig(X5, 2)
ref5 = PCA(2).fit(X5)

In [ ]:
check_close(variances, ref5.explained_variance_, message="phương sai giải thích")
check_close(np.abs(components), np.abs(ref5.components_), tol=1e-6, message="thành phần chính (cho phép đổi dấu)")

**Bài 6 — Gaussian Naive Bayes.** `fit_gnb(X, y)` → dict mỗi lớp: prior, trung bình, phương sai (chia n, cộng `1e-9 · max phương sai` để ổn định như scikit-learn); `predict_proba_gnb(model, X)` dùng log-xác suất rồi chuẩn hoá (log-sum-exp).

In [ ]:
def fit_gnb(X, y):
    ### START CODE HERE ### (≈ 5 dòng)
    # STARTER: return {}
    eps = 1e-9 * X.var(axis=0).max()
    return {c: (np.mean(y == c), X[y == c].mean(axis=0), X[y == c].var(axis=0) + eps) for c in np.unique(y)}
    ### END CODE HERE ###


def predict_proba_gnb(model, X):
    ### START CODE HERE ### (≈ 6 dòng)
    # STARTER: return np.zeros((len(X), len(model)))
    log_post = []
    for c, (prior, mu, var) in sorted(model.items()):
        log_post.append(np.log(prior) - 0.5 * np.sum(np.log(2 * np.pi * var) + (X - mu) ** 2 / var, axis=1))
    L = np.array(log_post).T
    L -= L.max(axis=1, keepdims=True)
    return np.exp(L) / np.exp(L).sum(axis=1, keepdims=True)
    ### END CODE HERE ###


X6, y6 = make_classification_2d("blobs", n=300, noise=1.0, seed=3)

In [ ]:
check_close(predict_proba_gnb(fit_gnb(X6, y6), X6), GaussianNB().fit(X6, y6).predict_proba(X6), tol=1e-6, message="khớp scikit-learn")